<div style="width: 100%; height: 5px; background-color: green;"></div>
<b>Feeding a realtime synth from an Encodec sequence</b>

This is a step on our way to streaming neural nets that generate codec token frames. Here, we encode a file and then supply a backend synth with decodings on periodic callbacks. We do something analagus to the way audio signals are reconstructed from fourier transform frames. WE have "chunks" consisting of chunk_length frames, and a "hop size" that is smaller than a chunk. Chunks are decoded, but only the last "hop length" frames of audio are taken and appended to the audio reconstruction. NOTE that the audio need not be overlap-added because the decoding already aligns phases at frame boundaries due to the "receptive field" context-dependent reconstruction of audio from mulitple frames. 

<div style="width: 100%; height: 5px; background-color: green;"></div>
<b>Parameters</b>

In [ ]:
#import librosa
import soundfile as sf
import torch
import os
import numpy as np
from pathlib import Path
import copy # for deepcopy

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel, AutoProcessor

from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"

In [ ]:
import time

In [ ]:
# for the rt synth
from realtime_synth.generators.base import BaseGenerator
from realtime_synth.utils import exp_map01
from realtime_synth_ui import build_synth_ui

# import the system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

In [ ]:
data_dir = "wav24k"
audiopath = Path(data_dir, "DSBugs--busybodyFreqFactor-00.60--c-00--x-00.wav")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Parameters</b>

In [ ]:
#audiopath = data_dir / "DSBugs--busybodyFreqFactor-00.60--c-00--x-00.wav"
Kbs = 3  # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]
print(f" path is {audiopath}")
g_hopsize=12
g_chunksize=24
sr=24000

In [ ]:
#audio_array, sr = librosa.load(audiopath, sr=sr, mono=True)
audio_array, sr = sf.read(audiopath)

print(f"Length of audio array is {len(audio_array)}, and sr is {sr}")

print(f'Original')
display(Audio(audio_array, rate=sr))

In [ ]:
# load the model + processor (for pre-processing the audio)
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
model.eval()
model.config.target_bandwidths = [Kbs] # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz", use_fast=False)
model.device

In [ ]:
audio = audio_array.astype(np.float32, copy=False)   # ensure float32
# (A) Mono → [1, T] (channels-first, what torchaudio likes)
wav_t = torch.from_numpy(audio) #.unsqueeze(0) 

inputs = processor(raw_audio=wav_t, sampling_rate=sr, return_tensors="pt")
inputs["input_values"].shape

In [ ]:
# Encode the entire wave file (and then we will decode it segment by segment)
encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
#encoder_outputs[0].shape #returns a batch....torch.Size([1, 1, 4, 375])
encoder_outputs
encoder_outputs.audio_codes[0].shape
encoder_outputs

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>CodecSynth</b>

In [ ]:
def prepend_warmup_segment(x: torch.Tensor, n: int) -> torch.Tensor:
    """
    Return a new tensor with the first `n` frames copied to the *beginning*.
    x can be [B, C, Q, T] (time last); preserves all leading dims.
    Result shape: [..., T+n]
    """
    n = int(max(0, n))
    if n == 0:
        return x
    T = x.size(-1)
    n = min(n, T)             # clamp
    head = x[..., :n]         # keep all leading dims
    return torch.cat([head, x], dim=-1)

In [ ]:
class MyEncodecPlayer(BaseGenerator):
    # normalized params in [0,1]
    param_labels = ["Freq (Hz)", "Amp", "fooparam"]

    # -----------------------
    def __init__(self, tokenseq,  buffersize, init_norm_params=None):
        super().__init__(init_norm_params or [0.5, 0.6, 0])  # defaults
        self.set_params(self.norm_params)  # initialize semantic values

        self.chunksizeframes=g_chunksize # decode this many frames each time
        self.framehopsize=g_hopsize # decode a new chunk every framehopsize
        self.nextendframe=self.framehopsize
        
        self.tokenseq=copy.deepcopy(tokenseq)
        self.buffersize=buffersize
        self.nextsample = 0
        self.framesizesamples=sr//75 # 75 (independent of g_chunksize) because encoder is 75 frames per second

        self.currentchunkframe=0 #nth frame in the chunk of audio we are playing

        self.warmup_len = self.chunksizeframes-self.framehopsize  # how many time steps to duplicate/append
        self.genaudioframe=0 #mth frame weve generated in total (considering warm up already generated)

        #Create extended codec list with warmup ----------------------
       
        
        with torch.no_grad():
            codes = self.tokenseq.audio_codes          # [B, C, Q, T]
            self.tokenseq.audio_codes = prepend_warmup_segment(codes, self.warmup_len)  # -> [B, C, Q, T+n]
        # --------------------------------------------------------------------------------------------------

        # now grab first chunk of audio (This is exactly the getNextChunk function, but we can't call it in init()!
        print(f"!!!!!!!!!! shape {self.tokenseq.audio_codes[..., self.genaudioframe:self.genaudioframe+self.chunksizeframes].shape}")
        with torch.inference_mode():
            #print(f" The thing we pass to model.decode has this shape: {self.tokenseq.audio_codes[..., self.genaudioframe:self.genaudioframe+self.chunksizeframes].shape}")
            self.thisaudioseq = model.decode(self.tokenseq.audio_codes[..., self.genaudioframe:self.genaudioframe+self.chunksizeframes], self.tokenseq.audio_scales, inputs["padding_mask"])[0]
        self.thisaudioseq=self.thisaudioseq[0, 0].detach().numpy()
        self.thisaudioseq=self.thisaudioseq[-self.framehopsize*self.framesizesamples:]
        
        
        self.nextaudioseq=None  #compute ahead of time, same till ready to use

        print(f"In init, length of extended codes is {self.tokenseq.audio_codes[0].shape[2]}, and the audio has a length of {len(self.thisaudioseq)}")
        self._callrecord=f""
        self._decodetime=0

    # -----------------------
    def getNextChunk(self):
        self.genaudioframe = self.genaudioframe+self.framehopsize

        self._callrecord = self._callrecord + f";(start: {self.genaudioframe}, end: {self.genaudioframe  + self.chunksizeframes})"
        
        #if we we don't have enough of a code seequence length, return 0's
        if self.genaudioframe  + self.chunksizeframes >= self.tokenseq.audio_codes[0].shape[2] :
            return  np.zeros(self.chunksizeframes*self.buffersize, dtype=np.float32)
            
        #decode a whole chunk 
        start_time = time.monotonic()
        with torch.inference_mode():
            nextseq = model.decode(self.tokenseq.audio_codes[..., self.genaudioframe:self.genaudioframe+self.chunksizeframes], self.tokenseq.audio_scales, inputs["padding_mask"])[0]
        nextseq=nextseq[0, 0].detach().numpy()
        self._decodetime = self._decodetime  + time.monotonic() - start_time
        
        #but take only the hopsize of audio that we need
        return nextseq[-self.framehopsize*self.framesizesamples:]
        
    # -----------------------   
    def set_params(self, norm_params):
        super().set_params(norm_params)
        # Map [0,1] → semantic values
        self.freq = float(exp_map01(self.norm_params[0], 20.0, 2000.0))  # exponential Hz
        self.amp  = float(self.norm_params[1])                           # linear gain 0..1
        self.foo  = float(self.norm_params[2]) 

    # -----------------------
    def generate(self, frames, sr):
       
        assert frames == self.buffersize, "ooh, you're in trouble if frames requested is different than the buffer size."

        if self.amp <= 0.0 or self.freq <= 0.0 or self.freq < 0.0:
            return np.zeros(self.buffersize, dtype=np.float32)
            
        endsamp=self.nextsample+self.buffersize
        y = self.thisaudioseq[self.nextsample:endsamp]
        self.nextsample=endsamp

        #This is how to get error reports out of the generation thread! After Playing, int he following cell call:
        #      print(getattr(synth.gen, "_last_error", None))
        if self.currentchunkframe==0 :
            try:
                self.nextaudioseq = self.getNextChunk()   # fast, non-blocking work only
            except Exception as e:
                self._last_error = repr(e)
                return np.zeros(frames, dtype=np.float32)

        try:    
            self.currentchunkframe = self.currentchunkframe+1
            
            if self.currentchunkframe==self.framehopsize :
                 self.thisaudioseq = self.nextaudioseq # it should be waiting for us
                 self.currentchunkframe = 0
                 self.nextsample = 0
        except Exception as e:
            self._last_error = repr(e)
             
        return self.amp*y.astype(np.float32)

    # -----------------------
    def formatted_readouts(self):
        # Optional: pretty labels shown next to sliders
        return [f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[2]}: {self.foo:.3f}"]
        

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>ON LINE, Realtime TEST</b>

In [ ]:
GENS = {
    "MyEncodecPlayer": lambda: MyEncodecPlayer(encoder_outputs, buffersize),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print(f"Will use samplerate = {sr} and blocksize = {buffersize}")
synth, ui = build_synth_ui(GENS, samplerate=sr, blocksize=buffersize, channels=1)

In [ ]:
print(getattr(synth.gen, "_last_error", None))

In [ ]:
print(getattr(synth.gen, "_callrecord ", None))

In [ ]:
print(getattr(synth.gen, "__decodetime", None))

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST</b>

In [ ]:
# just checking, not passed to to the synth which takes classes
foo= MyEncodecPlayer(encoder_outputs, buffersize)
foo.thisaudioseq
len(foo.thisaudioseq)
#display(Audio(foo.thisaudioseq, rate=sr))

In [ ]:
# chunk=foo.thisaudioseq
# for i in range(0,1) :
#     foo.getNextChunk()
#     chunk = np.concatenate((chunk, foo.thisaudioseq), axis=0) 
# len(chunk)

In [ ]:
c=[]
secs=5
fps=75
hops=int(secs*fps/g_hopsize)


for hopnum in range(0,hops) :
    c=  np.concatenate((c , foo.thisaudioseq), axis=0) 
    foo.thisaudioseq=foo.getNextChunk()
print(f"time spent decoding = {foo._decodetime:.2f}")


In [ ]:
display(Audio(c, rate=sr))

In [ ]:
print(f"{foo._callrecord}")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST 2 - calling generate</b>

In [ ]:
bar= MyEncodecPlayer(encoder_outputs, buffersize)

In [ ]:
audio_out=[]
#bar._decodetime=0
for buffernum in range(0,375) :
    audio_out =  np.concatenate((audio_out, bar.generate(buffersize, sr)), axis=0) 
print(f"time spent decoding = {bar._decodetime:.2f}")
display(Audio(audio_out, rate=sr))    